# Binary Classification of Government-Related Funding Organizations

This notebook uses SVM for binary classification of organization names.

- Class 0: Non-government-related funding organization
- Class 1: Government-related funding organization
- Evaluation method: Stratified 10-fold cross-validation
- Input feature: `Org_Name`
- Target variable: `Class`

## 1. Install Required Packages

In [1]:
# Install the packages required for SVM-based text classification.
%pip install -q --upgrade \
    scikit-learn \
    pandas \
    numpy \
    scipy \
    joblib \
    openpyxl

Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

In [1]:
# =============================================================================
# Import Libraries
# =============================================================================

import json
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

## 3. Experiment Configuration

In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

DATA_FILE = Path("./org_name_LIS.csv")

TEXT_COLUMN = "Org_Name"
LABEL_COLUMN = "Class"
ID_COLUMN = "Instance_ID"

MODEL_CHOICE = "svm"
MODEL_NAME = "TF-IDF + Calibrated Linear SVM"

RANDOM_SEED = 42
N_SPLITS = 10

ID2LABEL = {
    0: "non-gov",
    1: "gov"
}

LABEL2ID = {
    "non-gov": 0,
    "gov": 1
}

# -------------------------------------------------------------------------
# TF-IDF settings
# -------------------------------------------------------------------------

TFIDF_ANALYZER = "char_wb"
TFIDF_NGRAM_RANGE = (2, 5)
TFIDF_MIN_DF = 1
TFIDF_MAX_DF = 1.0
TFIDF_SUBLINEAR_TF = True
TFIDF_MAX_FEATURES = None

# -------------------------------------------------------------------------
# SVM settings
# -------------------------------------------------------------------------

SVM_C = 1.0
SVM_CLASS_WEIGHT = None

# Number of internal folds used to calibrate SVM probabilities.
CALIBRATION_CV = 5
CALIBRATION_METHOD = "sigmoid"

# -------------------------------------------------------------------------
# Output directories
# -------------------------------------------------------------------------

OUTPUT_ROOT = Path("./organization_classification_results_svm")
RESULTS_DIR = OUTPUT_ROOT / "evaluation_results"
FINAL_MODEL_DIR = OUTPUT_ROOT / "final_model"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATA_FILE}")
print(f"Model: {MODEL_NAME}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Final model directory: {FINAL_MODEL_DIR}")

Dataset: org_name_LIS.csv
Model: TF-IDF + Calibrated Linear SVM
Results directory: organization_classification_results_svm/evaluation_results
Final model directory: organization_classification_results_svm/final_model


## 4. Set Random Seeds

In [3]:
# =============================================================================
# Set Random Seeds
# =============================================================================

def set_seed(seed: int) -> None:
    """Set Python and NumPy random seeds."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

## 5. Load and Validate the Dataset

In [4]:
# =============================================================================
# Load Dataset
# =============================================================================

df = pd.read_csv(
    DATA_FILE,
    encoding="utf-8-sig"
)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

display(df.head())

Dataset shape: (500, 2)
Columns: ['Org_Name', 'Class']


,Org_Name,Class
0,Sapienza Università di Roma,0
1,Office of Science of the United States Departm...,1
2,Indiana University,0
3,University of International Business and Econo...,0
4,The Innovation of Foundation of the First Affi...,0


In [5]:
# =============================================================================
# Validate Dataset
# =============================================================================

required_columns = {
    TEXT_COLUMN,
    LABEL_COLUMN
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

# Remove rows with missing text or labels.
df = df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
).reset_index(drop=True)

# Normalize basic text formatting without changing accents.
df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.strip()
)

# Convert the target variable to integer.
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

# Add a stable instance identifier.
df.insert(
    0,
    ID_COLUMN,
    np.arange(1, len(df) + 1)
)

valid_labels = {0, 1}
observed_labels = set(df[LABEL_COLUMN].unique())

if not observed_labels.issubset(valid_labels):
    raise ValueError(
        f"Unexpected class labels found: {observed_labels}"
    )

if df[TEXT_COLUMN].eq("").any():
    raise ValueError(
        "At least one organization name is empty."
    )

if df[TEXT_COLUMN].duplicated().any():
    print(
        "Warning: Duplicate organization names were detected."
    )

print(f"Validated instances: {len(df):,}")

print("\nClass distribution:")
print(
    df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
)

display(df.head())

Validated instances: 500

Class distribution:
Class
0    255
1    245
Name: count, dtype: int64


,Instance_ID,Org_Name,Class
0,1,Sapienza Università di Roma,0
1,2,Office of Science of the United States Departm...,1
2,3,Indiana University,0
3,4,University of International Business and Econo...,0
4,5,The Innovation of Foundation of the First Affi...,0


## 6. Define the SVM Pipeline

In [6]:
# =============================================================================
# Define the SVM Pipeline
# =============================================================================

def initialize_svm_pipeline(
    random_seed: int
) -> Pipeline:
    """
    Initialize a fresh TF-IDF and calibrated linear SVM pipeline.

    The vectorizer and classifier are fitted using only the training
    portion of each cross-validation fold.
    """

    vectorizer = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        max_features=TFIDF_MAX_FEATURES,
        lowercase=True,
        strip_accents=None,
        dtype=np.float64
    )

    base_svm = LinearSVC(
        C=SVM_C,
        class_weight=SVM_CLASS_WEIGHT,
        random_state=random_seed,
        max_iter=10000
    )

    # LinearSVC does not directly produce probability estimates.
    # Calibration converts decision scores into estimated probabilities.
    calibrated_svm = CalibratedClassifierCV(
        estimator=base_svm,
        method=CALIBRATION_METHOD,
        cv=CALIBRATION_CV
    )

    pipeline = Pipeline(
        steps=[
            ("tfidf", vectorizer),
            ("classifier", calibrated_svm)
        ]
    )

    return pipeline

## 7. Define Evaluation Function

In [7]:
# =============================================================================
# Define Fold-Level Evaluation Function
# =============================================================================

def calculate_fold_metrics(
    true_labels: np.ndarray,
    predicted_labels: np.ndarray,
    fold_number: int
) -> dict:
    """Calculate binary and aggregate classification metrics."""

    accuracy = accuracy_score(
        true_labels,
        predicted_labels
    )

    class_precision, class_recall, class_f1, class_support = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average=None,
            zero_division=0
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average="weighted",
            zero_division=0
        )
    )

    return {
        "fold": fold_number,
        "accuracy": float(accuracy),

        "class_0_precision": float(class_precision[0]),
        "class_0_recall": float(class_recall[0]),
        "class_0_f1": float(class_f1[0]),
        "class_0_support": int(class_support[0]),

        "class_1_precision": float(class_precision[1]),
        "class_1_recall": float(class_recall[1]),
        "class_1_f1": float(class_f1[1]),
        "class_1_support": int(class_support[1]),

        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),

        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1)
    }

## 8. Configure Stratified 10-Fold Cross-Validation

In [8]:
# =============================================================================
# Configure Stratified 10-Fold Cross-Validation
# =============================================================================

texts = df[TEXT_COLUMN].to_numpy()
labels = df[LABEL_COLUMN].to_numpy()

stratified_kfold = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

fold_metrics_list = []
confusion_matrices = []
out_of_fold_predictions = []

print(
    f"Configured stratified {N_SPLITS}-fold cross-validation."
)

Configured stratified 10-fold cross-validation.


## 9. Run Stratified 10-Fold Cross-Validation

In [9]:
# =============================================================================
# Main Cross-Validation Loop
# =============================================================================

for fold_number, (train_indices, validation_indices) in enumerate(
    stratified_kfold.split(texts, labels),
    start=1
):
    print("=" * 100)
    print(f"Starting fold {fold_number} of {N_SPLITS}")
    print("=" * 100)

    fold_seed = RANDOM_SEED + fold_number
    set_seed(fold_seed)

    # -------------------------------------------------------------------------
    # Create training and validation data
    # -------------------------------------------------------------------------

    train_df = df.iloc[train_indices].copy()
    validation_df = df.iloc[validation_indices].copy()

    train_texts = train_df[TEXT_COLUMN].to_numpy()
    train_labels = train_df[LABEL_COLUMN].to_numpy()

    validation_texts = validation_df[TEXT_COLUMN].to_numpy()
    true_labels = validation_df[LABEL_COLUMN].to_numpy()

    print(f"Training instances: {len(train_df):,}")
    print(f"Validation instances: {len(validation_df):,}")

    print("\nTraining class distribution:")
    print(
        train_df[LABEL_COLUMN]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print("Validation class distribution:")
    print(
        validation_df[LABEL_COLUMN]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    # -------------------------------------------------------------------------
    # Initialize and train a fresh model for the fold
    # -------------------------------------------------------------------------

    svm_pipeline = initialize_svm_pipeline(
        random_seed=fold_seed
    )

    svm_pipeline.fit(
        train_texts,
        train_labels
    )

    # -------------------------------------------------------------------------
    # Generate class predictions and calibrated probabilities
    # -------------------------------------------------------------------------

    fold_predictions = svm_pipeline.predict(
        validation_texts
    )

    fold_probabilities = svm_pipeline.predict_proba(
        validation_texts
    )

    # Ensure probability columns correspond to classes 0 and 1.
    pipeline_classes = (
        svm_pipeline
        .named_steps["classifier"]
        .classes_
    )

    if not np.array_equal(
        pipeline_classes,
        np.array([0, 1])
    ):
        raise RuntimeError(
            f"Unexpected classifier class order: {pipeline_classes}"
        )

    # -------------------------------------------------------------------------
    # Calculate fold-level metrics
    # -------------------------------------------------------------------------

    fold_metrics = calculate_fold_metrics(
        true_labels=true_labels,
        predicted_labels=fold_predictions,
        fold_number=fold_number
    )

    fold_metrics_list.append(fold_metrics)

    # -------------------------------------------------------------------------
    # Calculate and store the confusion matrix
    # -------------------------------------------------------------------------

    fold_confusion_matrix = confusion_matrix(
        true_labels,
        fold_predictions,
        labels=[0, 1]
    )

    confusion_matrices.append(
        {
            "fold": fold_number,
            "true_0_pred_0": int(
                fold_confusion_matrix[0, 0]
            ),
            "true_0_pred_1": int(
                fold_confusion_matrix[0, 1]
            ),
            "true_1_pred_0": int(
                fold_confusion_matrix[1, 0]
            ),
            "true_1_pred_1": int(
                fold_confusion_matrix[1, 1]
            )
        }
    )

    # -------------------------------------------------------------------------
    # Store instance-level out-of-fold predictions
    # -------------------------------------------------------------------------

    for local_position, original_index in enumerate(
        validation_indices
    ):
        original_row = df.iloc[original_index]

        predicted_class = int(
            fold_predictions[local_position]
        )

        true_class = int(
            true_labels[local_position]
        )

        out_of_fold_predictions.append(
            {
                ID_COLUMN: int(
                    original_row[ID_COLUMN]
                ),
                TEXT_COLUMN: original_row[TEXT_COLUMN],

                "True_Class": true_class,
                "True_Label": ID2LABEL[true_class],

                "Predicted_Class": predicted_class,
                "Predicted_Label": ID2LABEL[
                    predicted_class
                ],

                "Probability_Class_0_non_gov": float(
                    fold_probabilities[
                        local_position,
                        0
                    ]
                ),
                "Probability_Class_1_gov": float(
                    fold_probabilities[
                        local_position,
                        1
                    ]
                ),

                "Prediction_Confidence": float(
                    np.max(
                        fold_probabilities[
                            local_position
                        ]
                    )
                ),

                "Correct_Prediction": bool(
                    predicted_class == true_class
                ),

                "Fold": fold_number,
                "Model_Choice": MODEL_CHOICE,
                "Model_Name": MODEL_NAME
            }
        )

    # -------------------------------------------------------------------------
    # Display current fold results
    # -------------------------------------------------------------------------

    print(f"\nFold {fold_number} results:")

    for metric_name, metric_value in fold_metrics.items():
        if isinstance(
            metric_value,
            (float, np.floating)
        ):
            print(
                f"{metric_name}: "
                f"{metric_value:.4f}"
            )

    print("\nConfusion matrix:")
    print(fold_confusion_matrix)

    # The fold model is not retained.
    del svm_pipeline

print("\nCross-validation completed.")

Starting fold 1 of 10
Training instances: 450
Validation instances: 50

Training class distribution:
{0: 229, 1: 221}
Validation class distribution:
{0: 26, 1: 24}

Fold 1 results:
accuracy: 0.8400
class_0_precision: 0.8214
class_0_recall: 0.8846
class_0_f1: 0.8519
class_1_precision: 0.8636
class_1_recall: 0.7917
class_1_f1: 0.8261
macro_precision: 0.8425
macro_recall: 0.8381
macro_f1: 0.8390
weighted_precision: 0.8417
weighted_recall: 0.8400
weighted_f1: 0.8395

Confusion matrix:
[[23  3]
 [ 5 19]]
Starting fold 2 of 10
Training instances: 450
Validation instances: 50

Training class distribution:
{0: 229, 1: 221}
Validation class distribution:
{0: 26, 1: 24}

Fold 2 results:
accuracy: 0.8200
class_0_precision: 0.8400
class_0_recall: 0.8077
class_0_f1: 0.8235
class_1_precision: 0.8000
class_1_recall: 0.8333
class_1_f1: 0.8163
macro_precision: 0.8200
macro_recall: 0.8205
macro_f1: 0.8199
weighted_precision: 0.8208
weighted_recall: 0.8200
weighted_f1: 0.8201

Confusion matrix:
[[21  5]


## 10. Save Fold-Level Evaluation Results

In [10]:
# =============================================================================
# Save Fold-Level Evaluation Results
# =============================================================================

fold_metrics_df = (
    pd.DataFrame(fold_metrics_list)
    .sort_values("fold")
)

fold_metrics_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_metrics.csv"
)

fold_metrics_df.to_csv(
    fold_metrics_file,
    index=False,
    encoding="utf-8-sig"
)

display(fold_metrics_df)

print(
    f"Fold-level metrics saved to: "
    f"{fold_metrics_file}"
)

,fold,accuracy,class_0_precision,class_0_recall,class_0_f1,class_0_support,class_1_precision,class_1_recall,class_1_f1,class_1_support,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,1,0.84,0.821429,0.884615,0.851852,26,0.863636,0.791667,0.826087,24,0.842532,0.838141,0.838969,0.841688,0.84,0.839485
1,2,0.82,0.840000,0.807692,0.823529,26,0.800000,0.833333,0.816327,24,0.820000,0.820513,0.819928,0.820800,0.82,0.820072
2,3,0.88,0.857143,0.923077,0.888889,26,0.909091,0.833333,0.869565,24,0.883117,0.878205,0.879227,0.882078,0.88,0.879614
3,4,0.84,0.800000,0.923077,0.857143,26,0.900000,0.750000,0.818182,24,0.850000,0.836538,0.837662,0.848000,0.84,0.838442
4,5,0.78,0.894737,0.653846,0.755556,26,0.709677,0.916667,0.800000,24,0.802207,0.785256,0.777778,0.805908,0.78,0.776889
5,6,0.80,0.777778,0.840000,0.807692,25,0.826087,0.760000,0.791667,25,0.801932,0.800000,0.799679,0.801932,0.80,0.799679
6,7,0.86,0.909091,0.800000,0.851064,25,0.821429,0.920000,0.867925,25,0.865260,0.860000,0.859494,0.865260,0.86,0.859494
7,8,0.90,1.000000,0.800000,0.888889,25,0.833333,1.000000,0.909091,25,0.916667,0.900000,0.898990,0.916667,0.90,0.898990
8,9,0.78,0.818182,0.720000,0.765957,25,0.750000,0.840000,0.792453,25,0.784091,0.780000,0.779205,0.784091,0.78,0.779205
9,10,0.84,0.869565,0.800000,0.833333,25,0.814815,0.880000,0.846154,25,0.842190,0.840000,0.839744,0.842190,0.84,0.839744


Fold-level metrics saved to: organization_classification_results_svm/evaluation_results/svm_10fold_metrics.csv


## 11. Calculate Mean and Standard Deviation

In [11]:
# =============================================================================
# Aggregate Metrics
# =============================================================================

metric_columns = [
    "accuracy",

    "class_0_precision",
    "class_0_recall",
    "class_0_f1",

    "class_1_precision",
    "class_1_recall",
    "class_1_f1",

    "macro_precision",
    "macro_recall",
    "macro_f1",

    "weighted_precision",
    "weighted_recall",
    "weighted_f1"
]

aggregate_metrics_df = pd.DataFrame(
    {
        "Metric": metric_columns,

        "Mean": [
            fold_metrics_df[column].mean()
            for column in metric_columns
        ],

        "Standard_Deviation": [
            fold_metrics_df[column].std(ddof=1)
            for column in metric_columns
        ],

        "Minimum": [
            fold_metrics_df[column].min()
            for column in metric_columns
        ],

        "Maximum": [
            fold_metrics_df[column].max()
            for column in metric_columns
        ]
    }
)

aggregate_metrics_df[
    [
        "Mean",
        "Standard_Deviation",
        "Minimum",
        "Maximum"
    ]
] = aggregate_metrics_df[
    [
        "Mean",
        "Standard_Deviation",
        "Minimum",
        "Maximum"
    ]
].round(4)

aggregate_metrics_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_aggregate_metrics.csv"
)

aggregate_metrics_df.to_csv(
    aggregate_metrics_file,
    index=False,
    encoding="utf-8-sig"
)

display(aggregate_metrics_df)

print(
    f"Aggregate metrics saved to: "
    f"{aggregate_metrics_file}"
)

,Metric,Mean,Standard_Deviation,Minimum,Maximum
0,accuracy,0.8340,0.0401,0.7800,0.9000
1,class_0_precision,0.8588,0.0644,0.7778,1.0000
2,class_0_recall,0.8152,0.0847,0.6538,0.9231
3,class_0_f1,0.8324,0.0456,0.7556,0.8889
4,class_1_precision,0.8228,0.0614,0.7097,0.9091
5,class_1_recall,0.8525,0.0781,0.7500,1.0000
6,class_1_f1,0.8337,0.0387,0.7917,0.9091
7,macro_precision,0.8408,0.0407,0.7841,0.9167
8,macro_recall,0.8339,0.0390,0.7800,0.9000
9,macro_f1,0.8331,0.0402,0.7778,0.8990


Aggregate metrics saved to: organization_classification_results_svm/evaluation_results/svm_10fold_aggregate_metrics.csv


## 12. Save Fold-Level Confusion Matrices

In [12]:
# =============================================================================
# Save Fold-Level Confusion Matrices
# =============================================================================

confusion_matrices_df = (
    pd.DataFrame(confusion_matrices)
    .sort_values("fold")
)

confusion_matrix_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_confusion_matrices.csv"
)

confusion_matrices_df.to_csv(
    confusion_matrix_file,
    index=False,
    encoding="utf-8-sig"
)

display(confusion_matrices_df)

print(
    f"Confusion matrices saved to: "
    f"{confusion_matrix_file}"
)

,fold,true_0_pred_0,true_0_pred_1,true_1_pred_0,true_1_pred_1
0,1,23,3,5,19
1,2,21,5,4,20
2,3,24,2,4,20
3,4,24,2,6,18
4,5,17,9,2,22
5,6,21,4,6,19
6,7,20,5,2,23
7,8,20,5,0,25
8,9,18,7,4,21
9,10,20,5,3,22


Confusion matrices saved to: organization_classification_results_svm/evaluation_results/svm_10fold_confusion_matrices.csv


## 13. Save Instance-Level Out-of-Fold Predictions

In [13]:
# =============================================================================
# Save Out-of-Fold Predictions and Probabilities
# =============================================================================

out_of_fold_predictions_df = (
    pd.DataFrame(out_of_fold_predictions)
    .sort_values(ID_COLUMN)
)

if len(out_of_fold_predictions_df) != len(df):
    raise RuntimeError(
        "The number of out-of-fold predictions does not "
        "match the number of original instances."
    )

if out_of_fold_predictions_df[
    ID_COLUMN
].duplicated().any():
    raise RuntimeError(
        "At least one instance received more than one "
        "out-of-fold prediction."
    )

prediction_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_instance_predictions.csv"
)

out_of_fold_predictions_df.to_csv(
    prediction_file,
    index=False,
    encoding="utf-8-sig"
)

display(
    out_of_fold_predictions_df.head(20)
)

print(
    f"Out-of-fold predictions saved to: "
    f"{prediction_file}"
)

,Instance_ID,Org_Name,True_Class,True_Label,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence,Correct_Prediction,Fold,Model_Choice,Model_Name
350,1,Sapienza Università di Roma,0,non-gov,0,non-gov,0.960324,0.039676,0.960324,True,8,svm,TF-IDF + Calibrated Linear SVM
100,2,Office of Science of the United States Departm...,1,gov,0,non-gov,0.738991,0.261009,0.738991,False,3,svm,TF-IDF + Calibrated Linear SVM
0,3,Indiana University,0,non-gov,0,non-gov,0.993898,0.006102,0.993898,True,1,svm,TF-IDF + Calibrated Linear SVM
150,4,University of International Business and Econo...,0,non-gov,0,non-gov,0.952683,0.047317,0.952683,True,4,svm,TF-IDF + Calibrated Linear SVM
400,5,The Innovation of Foundation of the First Affi...,0,non-gov,0,non-gov,0.933260,0.066740,0.933260,True,9,svm,TF-IDF + Calibrated Linear SVM
200,6,Chanté St Clair Inglis,0,non-gov,1,gov,0.453802,0.546198,0.546198,False,5,svm,TF-IDF + Calibrated Linear SVM
351,7,Paramadina University,0,non-gov,0,non-gov,0.992431,0.007569,0.992431,True,8,svm,TF-IDF + Calibrated Linear SVM
151,8,"Sobolev Institute of Mathematics, Siberian Bra...",1,gov,0,non-gov,0.720408,0.279592,0.720408,False,4,svm,TF-IDF + Calibrated Linear SVM
352,9,"Horace H. Rackham School of Graduate Studies, ...",0,non-gov,0,non-gov,0.979469,0.020531,0.979469,True,8,svm,TF-IDF + Calibrated Linear SVM
450,10,Ministero Italiano per l’Università e la Ricerca,1,gov,1,gov,0.410301,0.589699,0.589699,True,10,svm,TF-IDF + Calibrated Linear SVM


Out-of-fold predictions saved to: organization_classification_results_svm/evaluation_results/svm_10fold_instance_predictions.csv


## 14. Calculate Overall Out-of-Fold Performance

In [14]:
# =============================================================================
# Pooled Out-of-Fold Evaluation
# =============================================================================

oof_true_labels = out_of_fold_predictions_df[
    "True_Class"
].to_numpy()

oof_predicted_labels = out_of_fold_predictions_df[
    "Predicted_Class"
].to_numpy()

oof_report_dictionary = classification_report(
    oof_true_labels,
    oof_predicted_labels,
    labels=[0, 1],
    target_names=["non-gov", "gov"],
    output_dict=True,
    zero_division=0
)

oof_report_text = classification_report(
    oof_true_labels,
    oof_predicted_labels,
    labels=[0, 1],
    target_names=["non-gov", "gov"],
    digits=4,
    zero_division=0
)

oof_confusion_matrix = confusion_matrix(
    oof_true_labels,
    oof_predicted_labels,
    labels=[0, 1]
)

print("Overall out-of-fold classification report:")
print(oof_report_text)

print("Overall out-of-fold confusion matrix:")
print(oof_confusion_matrix)

Overall out-of-fold classification report:
              precision    recall  f1-score   support

     non-gov     0.8525    0.8157    0.8337       255
         gov     0.8164    0.8531    0.8343       245

    accuracy                         0.8340       500
   macro avg     0.8344    0.8344    0.8340       500
weighted avg     0.8348    0.8340    0.8340       500

Overall out-of-fold confusion matrix:
[[208  47]
 [ 36 209]]


In [15]:
# =============================================================================
# Save Pooled Out-of-Fold Evaluation
# =============================================================================

oof_report_df = (
    pd.DataFrame(oof_report_dictionary)
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

oof_report_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_overall_oof_classification_report.csv"
)

oof_report_df.to_csv(
    oof_report_file,
    index=False,
    encoding="utf-8-sig"
)

oof_report_text_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_overall_oof_classification_report.txt"
)

with open(
    oof_report_text_file,
    "w",
    encoding="utf-8"
) as file:
    file.write(oof_report_text)

oof_confusion_matrix_df = pd.DataFrame(
    oof_confusion_matrix,
    index=[
        "True_0_non_gov",
        "True_1_gov"
    ],
    columns=[
        "Predicted_0_non_gov",
        "Predicted_1_gov"
    ]
)

oof_confusion_matrix_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_overall_oof_confusion_matrix.csv"
)

oof_confusion_matrix_df.to_csv(
    oof_confusion_matrix_file,
    encoding="utf-8-sig"
)

print(f"Saved: {oof_report_file}")
print(f"Saved: {oof_report_text_file}")
print(f"Saved: {oof_confusion_matrix_file}")

Saved: organization_classification_results_svm/evaluation_results/svm_overall_oof_classification_report.csv
Saved: organization_classification_results_svm/evaluation_results/svm_overall_oof_classification_report.txt
Saved: organization_classification_results_svm/evaluation_results/svm_overall_oof_confusion_matrix.csv


## 15. Save Misclassified Instances

In [16]:
# =============================================================================
# Misclassification Analysis
# =============================================================================

misclassified_df = out_of_fold_predictions_df[
    ~out_of_fold_predictions_df[
        "Correct_Prediction"
    ]
].copy()

misclassified_df = misclassified_df.sort_values(
    "Prediction_Confidence",
    ascending=False
)

misclassified_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_misclassified_instances.csv"
)

misclassified_df.to_csv(
    misclassified_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Number of misclassified instances: "
    f"{len(misclassified_df):,}"
)

display(
    misclassified_df.head(30)
)

print(
    f"Misclassified instances saved to: "
    f"{misclassified_file}"
)

Number of misclassified instances: 83


,Instance_ID,Org_Name,True_Class,True_Label,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence,Correct_Prediction,Fold,Model_Choice,Model_Name
19,190,Institute for Research in Fundamental Sciences,0,non-gov,1,gov,0.008571,0.991429,0.991429,False,1,svm,TF-IDF + Calibrated Linear SVM
499,500,Double First Class University Plan,1,gov,0,non-gov,0.984610,0.015390,0.984610,False,10,svm,TF-IDF + Calibrated Linear SVM
50,16,African Center of Excellence for Innovative Te...,0,non-gov,1,gov,0.071540,0.928460,0.928460,False,2,svm,TF-IDF + Calibrated Linear SVM
339,430,Consejo Ciudadano para la Seguridad Pública y ...,0,non-gov,1,gov,0.075824,0.924176,0.924176,False,7,svm,TF-IDF + Calibrated Linear SVM
31,301,SystemsX.ch,1,gov,0,non-gov,0.900829,0.099171,0.900829,False,1,svm,TF-IDF + Calibrated Linear SVM
188,373,Eastern and Central African regional branch of...,0,non-gov,1,gov,0.115479,0.884521,0.884521,False,4,svm,TF-IDF + Calibrated Linear SVM
396,472,Leibniz Programme for Women Professors,0,non-gov,1,gov,0.123338,0.876662,0.876662,False,8,svm,TF-IDF + Calibrated Linear SVM
64,196,Internal Security Fund - Police,1,gov,0,non-gov,0.872724,0.127276,0.872724,False,2,svm,TF-IDF + Calibrated Linear SVM
258,111,Penn State Berks Office of Planning,0,non-gov,1,gov,0.128349,0.871651,0.871651,False,6,svm,TF-IDF + Calibrated Linear SVM
296,459,Office of the United Nations High Commissioner...,0,non-gov,1,gov,0.152807,0.847193,0.847193,False,6,svm,TF-IDF + Calibrated Linear SVM


Misclassified instances saved to: organization_classification_results_svm/evaluation_results/svm_misclassified_instances.csv


## 16. Train the Final Deployment Model

In [17]:
# =============================================================================
# Train the Final Model on the Full Dataset
# =============================================================================

set_seed(RANDOM_SEED)

final_svm_pipeline = initialize_svm_pipeline(
    random_seed=RANDOM_SEED
)

final_svm_pipeline.fit(
    df[TEXT_COLUMN].to_numpy(),
    df[LABEL_COLUMN].to_numpy()
)

print(
    "Final full-data SVM model training completed."
)

Final full-data SVM model training completed.


## 17. Save Final Deployment Model

In [18]:
# =============================================================================
# Save Final Deployment Model
# =============================================================================

deployment_model_file = (
    FINAL_MODEL_DIR
    / f"{MODEL_CHOICE}_deployment_pipeline.joblib"
)

joblib.dump(
    final_svm_pipeline,
    deployment_model_file
)

experiment_metadata = {
    "model_choice": MODEL_CHOICE,
    "model_name": MODEL_NAME,

    "text_column": TEXT_COLUMN,
    "label_column": LABEL_COLUMN,

    "number_of_training_instances": int(len(df)),
    "number_of_classes": 2,

    "id2label": {
        str(key): value
        for key, value in ID2LABEL.items()
    },

    "label2id": LABEL2ID,

    "tfidf_analyzer": TFIDF_ANALYZER,
    "tfidf_ngram_range": list(
        TFIDF_NGRAM_RANGE
    ),
    "tfidf_min_df": TFIDF_MIN_DF,
    "tfidf_max_df": TFIDF_MAX_DF,
    "tfidf_sublinear_tf": TFIDF_SUBLINEAR_TF,
    "tfidf_max_features": TFIDF_MAX_FEATURES,

    "svm_c": SVM_C,
    "svm_class_weight": SVM_CLASS_WEIGHT,

    "calibration_method": CALIBRATION_METHOD,
    "calibration_cv": CALIBRATION_CV,

    "random_seed": RANDOM_SEED,
    "cross_validation_folds": N_SPLITS,

    "scikit_learn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "python_version": sys.version
}

metadata_file = (
    FINAL_MODEL_DIR
    / "experiment_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Final deployment pipeline saved to: "
    f"{deployment_model_file}"
)

print(
    f"Experiment metadata saved to: "
    f"{metadata_file}"
)

Final deployment pipeline saved to: organization_classification_results_svm/final_model/svm_deployment_pipeline.joblib
Experiment metadata saved to: organization_classification_results_svm/final_model/experiment_metadata.json


## 18. Reload and Test the Saved Model

In [19]:
# =============================================================================
# Reload the Saved Deployment Model
# =============================================================================

loaded_svm_pipeline = joblib.load(
    deployment_model_file
)

print(
    "Saved SVM deployment pipeline loaded successfully."
)

Saved SVM deployment pipeline loaded successfully.


In [20]:
# =============================================================================
# Define an Inference Function
# =============================================================================

def predict_organization(
    organization_name: str,
    model_pipeline: Pipeline = loaded_svm_pipeline
) -> dict:
    """Predict the government classification of one organization."""

    if not isinstance(
        organization_name,
        str
    ):
        raise TypeError(
            "organization_name must be a string."
        )

    cleaned_name = organization_name.strip()

    if not cleaned_name:
        raise ValueError(
            "organization_name cannot be empty."
        )

    predicted_class = int(
        model_pipeline.predict(
            [cleaned_name]
        )[0]
    )

    probabilities = (
        model_pipeline.predict_proba(
            [cleaned_name]
        )[0]
    )

    return {
        "Org_Name": cleaned_name,

        "Predicted_Class": predicted_class,
        "Predicted_Label": ID2LABEL[
            predicted_class
        ],

        "Probability_Class_0_non_gov": float(
            probabilities[0]
        ),

        "Probability_Class_1_gov": float(
            probabilities[1]
        ),

        "Prediction_Confidence": float(
            np.max(probabilities)
        )
    }

In [21]:
# Test the inference function.
test_prediction = predict_organization(
    "National Science Foundation"
)

test_prediction

{'Org_Name': 'National Science Foundation',
 'Predicted_Class': 1,
 'Predicted_Label': 'gov',
 'Probability_Class_0_non_gov': 0.1623361546044963,
 'Probability_Class_1_gov': 0.8376638453955036,
 'Prediction_Confidence': 0.8376638453955036}

## 19. Batch Prediction

In [22]:
# =============================================================================
# Batch Prediction Function
# =============================================================================

def predict_organization_batch(
    organization_names: list[str],
    model_pipeline: Pipeline = loaded_svm_pipeline
) -> pd.DataFrame:
    """Predict classifications for multiple organization names."""

    cleaned_names = [
        str(name).strip()
        for name in organization_names
    ]

    if any(
        name == ""
        for name in cleaned_names
    ):
        raise ValueError(
            "At least one organization name is empty."
        )

    predictions = model_pipeline.predict(
        cleaned_names
    )

    probabilities = model_pipeline.predict_proba(
        cleaned_names
    )

    results = []

    for index, organization_name in enumerate(
        cleaned_names
    ):
        predicted_class = int(
            predictions[index]
        )

        results.append(
            {
                TEXT_COLUMN: organization_name,

                "Predicted_Class": predicted_class,

                "Predicted_Label": ID2LABEL[
                    predicted_class
                ],

                "Probability_Class_0_non_gov": float(
                    probabilities[index, 0]
                ),

                "Probability_Class_1_gov": float(
                    probabilities[index, 1]
                ),

                "Prediction_Confidence": float(
                    np.max(probabilities[index])
                )
            }
        )

    return pd.DataFrame(results)

In [23]:
sample_organizations = [
    "National Institutes of Health",
    "Harvard University",
    "Samsung Electronics",
    "Ministry of Education"
]

batch_predictions_df = predict_organization_batch(
    sample_organizations
)

display(batch_predictions_df)

,Org_Name,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence
0,National Institutes of Health,1,gov,0.070569,0.929431,0.929431
1,Harvard University,0,non-gov,0.998922,0.001078,0.998922
2,Samsung Electronics,0,non-gov,0.880530,0.119470,0.880530
3,Ministry of Education,1,gov,0.008359,0.991641,0.991641


## 20. Export Experiment Summary

In [24]:
# =============================================================================
# Save a Human-Readable Experiment Summary
# =============================================================================

summary_lines = [
    "Organization Name Binary Classification Experiment",
    "=" * 60,

    f"Model choice: {MODEL_CHOICE}",
    f"Model name: {MODEL_NAME}",

    f"Number of instances: {len(df)}",
    f"Number of folds: {N_SPLITS}",

    f"Class 0 count: "
    f"{(df[LABEL_COLUMN] == 0).sum()}",

    f"Class 1 count: "
    f"{(df[LABEL_COLUMN] == 1).sum()}",

    f"TF-IDF analyzer: {TFIDF_ANALYZER}",
    f"TF-IDF n-gram range: {TFIDF_NGRAM_RANGE}",
    f"SVM C: {SVM_C}",
    f"Probability calibration: "
    f"{CALIBRATION_METHOD}, CV={CALIBRATION_CV}",

    "",
    "Mean 10-fold evaluation results",
    "-" * 60
]

for _, metric_row in aggregate_metrics_df.iterrows():
    summary_lines.append(
        f"{metric_row['Metric']}: "
        f"{metric_row['Mean']:.4f} "
        f"(SD = "
        f"{metric_row['Standard_Deviation']:.4f})"
    )

summary_text = "\n".join(
    summary_lines
)

summary_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_experiment_summary.txt"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as file:
    file.write(summary_text)

print(summary_text)

print(
    f"\nSummary saved to: "
    f"{summary_file}"
)

Organization Name Binary Classification Experiment
Model choice: svm
Model name: TF-IDF + Calibrated Linear SVM
Number of instances: 500
Number of folds: 10
Class 0 count: 255
Class 1 count: 245
TF-IDF analyzer: char_wb
TF-IDF n-gram range: (2, 5)
SVM C: 1.0
Probability calibration: sigmoid, CV=5

Mean 10-fold evaluation results
------------------------------------------------------------
accuracy: 0.8340 (SD = 0.0401)
class_0_precision: 0.8588 (SD = 0.0644)
class_0_recall: 0.8152 (SD = 0.0847)
class_0_f1: 0.8324 (SD = 0.0456)
class_1_precision: 0.8228 (SD = 0.0614)
class_1_recall: 0.8525 (SD = 0.0781)
class_1_f1: 0.8337 (SD = 0.0387)
macro_precision: 0.8408 (SD = 0.0407)
macro_recall: 0.8339 (SD = 0.0390)
macro_f1: 0.8331 (SD = 0.0402)
weighted_precision: 0.8409 (SD = 0.0401)
weighted_recall: 0.8340 (SD = 0.0401)
weighted_f1: 0.8332 (SD = 0.0404)

Summary saved to: organization_classification_results_svm/evaluation_results/svm_experiment_summary.txt
